# Experiment 05: CatBoost

In [1]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
import json
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score, 
    precision_recall_curve
)
from catboost import CatBoostClassifier, Pool
from sklearn.utils.class_weight import compute_sample_weight

import mlflow
import mlflow.catboost

# Настройки
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# Добавляем путь к src
sys.path.append('..')
sys.path.append(os.path.abspath('..'))

from src.data.preprocess import DataPreprocessor


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank_Churn")

<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1780183529764, experiment_id='1', last_update_time=1780183529764, lifecycle_stage='active', name='Bank_Churn', tags={}, trace_location=None, workspace='default'>

In [3]:
# Загрузка конфигурации
config_path = "../configs/config.yaml"
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Конфигурация загружена из config.yaml")
else:
    config = {
        'random_state': 42,
        'test_size': 0.2
    }
    print("Используется конфигурация по умолчанию")

RANDOM_STATE = config.get('random_state', 42)
TEST_SIZE = config.get('test_size', 0.2)

print(f"Random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE}")

Конфигурация загружена из config.yaml
Random state: 42
Test size: 0.2


In [4]:
# Загрузка и предобработка данных

# Загружаем сырые данные
df_raw = pd.read_csv('../data/raw/Churn_Modelling.csv')
print(f"Загружено {len(df_raw)} записей")
print(f"Исходные колонки: {list(df_raw.columns)}")

# Инициализируем препроцессор
preprocessor = DataPreprocessor()

# Обрабатываем данные (fit_scaler=True для обучения)
X_scaled, y, df_processed = preprocessor.preprocess(df_raw, fit_scaler=True)

# Сохраняем препроцессор для сервиса
os.makedirs('../artifacts', exist_ok=True)
preprocessor.save('../artifacts/preprocessor.pkl')
print("Препроцессор сохранён в artifacts/preprocessor.pkl")

print(f"\nПосле предобработки:")
print(f"  - Признаков: {X_scaled.shape[1]}")
print(f"  - Целевая переменная: {y.name}")
print(f"  - Распределение: {y.value_counts().to_dict()}")

INFO:src.data.preprocess:DataPreprocessor инициализирован
INFO:src.data.preprocess:Создано 21 признаков
INFO:src.data.preprocess:Данные очищены. Форма: (10000, 14)
INFO:src.data.preprocess:Scaler обучен и применён
INFO:src.data.preprocess:Препроцессор сохранён в ../artifacts/preprocessor.pkl


Загружено 10000 записей
Исходные колонки: ['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
Препроцессор сохранён в artifacts/preprocessor.pkl

После предобработки:
  - Признаков: 13
  - Целевая переменная: Exited
  - Распределение: {0: 7963, 1: 2037}


In [5]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\nРазделение данных:")
print(f"  - Train: {X_train.shape[0]} записей (отток: {y_train.mean():.2%})")
print(f"  - Test: {X_test.shape[0]} записей (отток: {y_test.mean():.2%})")


Разделение данных:
  - Train: 8000 записей (отток: 20.38%)
  - Test: 2000 записей (отток: 20.35%)


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Рассчитываем weight для класса меньшинства
total_neg = (y_train == 0).sum()  # 7963
total_pos = (y_train == 1).sum()  # 2037
scale_pos_weight = total_neg / total_pos  # 7963/2037 ≈ 3.91

# Параметры модели
params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'border_count': 128,
    'class_weights': [1, scale_pos_weight],
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    'early_stopping_rounds': 50,
    'verbose': 100,
    'random_seed': RANDOM_STATE
}

# Запуск эксперимента в MLflow
with mlflow.start_run(run_name="catboost_default_v1"):
    # Логирование параметров
    mlflow.log_params(params)
    
    # Обучение модели
    catboost_default = CatBoostClassifier(**params)
    catboost_default.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        verbose=100,
        plot=False
    )
    
    # Предсказания
    y_pred_cat = catboost_default.predict(X_test)
    y_pred_proba_cat = catboost_default.predict_proba(X_test)[:, 1]
    
    # Метрики
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred_cat),
        "precision": precision_score(y_test, y_pred_cat),
        "recall": recall_score(y_test, y_pred_cat),
        "f1": f1_score(y_test, y_pred_cat),
        "roc_auc": roc_auc_score(y_test, y_pred_proba_cat)
    }
    
    # Логирование метрик
    mlflow.log_metrics(metrics)
    
    # Сохраняем модель
    os.makedirs('../artifacts/models', exist_ok=True)
    catboost_default.save_model('../artifacts/models/catboost_default_v1.cbm')
    mlflow.log_artifact('../artifacts/models/catboost_default_v1.cbm')
    
    print(f"\nCatBoost результаты:")
    print(f"  - Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  - Precision: {metrics['precision']:.4f}")
    print(f"  - Recall:    {metrics['recall']:.4f}")
    print(f"  - F1-Score:  {metrics['f1']:.4f}")
    print(f"  - ROC-AUC:   {metrics['roc_auc']:.4f}")
    print(f"  - Лучшая итерация: {catboost_default.get_best_iteration()}")

0:	test: 0.8102239	best: 0.8102239 (0)	total: 157ms	remaining: 1m 18s
100:	test: 0.8610537	best: 0.8613159 (94)	total: 401ms	remaining: 1.58s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8613158613
bestIteration = 94

Shrink model to first 95 iterations.

CatBoost результаты:
  - Accuracy:  0.7950
  - Precision: 0.4975
  - Recall:    0.7371
  - F1-Score:  0.5941
  - ROC-AUC:   0.8613
  - Лучшая итерация: 94
🏃 View run catboost_default_v1 at: http://localhost:5000/#/experiments/1/runs/2b699e6784364cb795c7eeeff1e01a72
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [9]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import mlflow
import mlflow.catboost
import pandas as pd
import numpy as np
import joblib

# Рассчитываем базовый вес для класса меньшинства
total_neg = (y_train == 0).sum()
total_pos = (y_train == 1).sum()
default_scale_pos_weight = total_neg / total_pos

print(f"Базовый scale_pos_weight: {default_scale_pos_weight:.2f}")

# Параметры для поиска
param_grid = {
    'iterations': [300, 500, 700],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [64, 128, 255],
    'class_weights': [
        [1, 1],           # без учёта дисбаланса
        [1, 2],           # вес для класса 1 = 2
        [1, 4],           # вес для класса 1 = 4
        [1, 6],           # вес для класса 1 = 6
        [1, int(default_scale_pos_weight)]  # автоматический вес
    ]
}

print("Параметры для поиска:")
for param, values in param_grid.items():
    print(f"  - {param}: {values}")

with mlflow.start_run(run_name="catboost_gridsearch_balanced"):
    
    # Логирование параметров GridSearch
    mlflow.log_param("grid_search_params", str(param_grid))
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("scoring", "roc_auc")
    mlflow.log_param("default_scale_pos_weight", default_scale_pos_weight)
    
    # Создание и обучение GridSearchCV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    
    # Базовый классификатор CatBoost
    catboost_base = CatBoostClassifier(
        eval_metric='AUC',
        loss_function='Logloss',
        early_stopping_rounds=50,
        verbose=False,
        random_seed=RANDOM_STATE
    )
    
    from sklearn.model_selection import RandomizedSearchCV

    grid_search = RandomizedSearchCV(
        catboost_base,
        param_grid,
        n_iter=50,         
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    
    print("\nЗапуск Grid Search для CatBoost...")
    grid_search.fit(X_train, y_train)
    
    # Логирование лучших параметров
    best_params = grid_search.best_params_
    mlflow.log_params(best_params)
    
    # Логирование метрик
    mlflow.log_metric("best_cv_roc_auc", grid_search.best_score_)
    mlflow.log_metric("best_cv_std", grid_search.cv_results_['std_test_score'][grid_search.best_index_])
    
    # Сохранение всех результатов CV
    cv_results_df = pd.DataFrame(grid_search.cv_results_)
    cv_results_df.to_csv('../artifacts/catboost_gridsearch_cv_results.csv', index=False)
    mlflow.log_artifact('../artifacts/catboost_gridsearch_cv_results.csv')
    
    # Логирование лучшей модели
    best_catboost = grid_search.best_estimator_
    
    # Предсказания на тестовой выборке
    y_pred_best = best_catboost.predict(X_test)
    y_pred_proba_best = best_catboost.predict_proba(X_test)[:, 1]
    
    # Поиск оптимального порога
    from sklearn.metrics import precision_recall_curve
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba_best)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores[:-1])
    optimal_threshold = thresholds[optimal_idx]
    
    # Метрики на тесте (порог 0.5)
    test_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_best) if accuracy_score(y_test, y_pred_best) is not None else 0.0,
        "precision": precision_score(y_test, y_pred_best) if precision_score(y_test, y_pred_best) is not None else 0.0,
        "recall": recall_score(y_test, y_pred_best) if recall_score(y_test, y_pred_best) is not None else 0.0,
        "f1": f1_score(y_test, y_pred_best) if f1_score(y_test, y_pred_best) is not None else 0.0,
        "roc_auc": roc_auc_score(y_test, y_pred_proba_best) if roc_auc_score(y_test, y_pred_proba_best) is not None else 0.0,
        "best_iteration": best_catboost.get_best_iteration() if best_catboost.get_best_iteration() is not None else 0
    }
    
    # Метрики при оптимальном пороге
    y_pred_optimal = (y_pred_proba_best >= optimal_threshold).astype(int)
    optimal_metrics = {
        "recall_optimal": recall_score(y_test, y_pred_optimal),
        "precision_optimal": precision_score(y_test, y_pred_optimal),
        "f1_optimal": f1_score(y_test, y_pred_optimal),
        "optimal_threshold": optimal_threshold
    }
    
    mlflow.log_metrics(test_metrics)
    mlflow.log_metrics(optimal_metrics)
    
    
    # Сохранение модели в формате pkl (для совместимости)
    joblib.dump(best_catboost, '../artifacts/models/catboost_best.pkl')
    mlflow.log_artifact('../artifacts/models/catboost_best.pkl')
    
    # Сохранение конфига с оптимальным порогом
    import yaml
    threshold_config = {
        'optimal_threshold': float(optimal_threshold),
        'model_type': 'CatBoost',
        'metrics_at_threshold': {
            'recall': float(optimal_metrics['recall_optimal']),
            'precision': float(optimal_metrics['precision_optimal']),
            'f1': float(optimal_metrics['f1_optimal'])
        },
        'best_params': {k: v for k, v in best_params.items()}
    }
    
    with open('../artifacts/catboost_threshold_config.yaml', 'w') as f:
        yaml.dump(threshold_config, f)
    mlflow.log_artifact('../artifacts/catboost_threshold_config.yaml')
    
    # Вывод результатов
    for param, value in best_params.items():
        print(f"  - {param}: {value}")
    print(f"\nЛучший CV ROC-AUC: {grid_search.best_score_:.4f}")
    
    print(f"\nМетрики на тестовой выборке (порог 0.5):")
    print(f"  - Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  - Precision: {test_metrics['precision']:.4f}")
    print(f"  - Recall:    {test_metrics['recall']:.4f}")
    print(f"  - F1-Score:  {test_metrics['f1']:.4f}")
    print(f"  - ROC-AUC:   {test_metrics['roc_auc']:.4f}")
    print(f"  - Итераций:  {test_metrics['best_iteration']}")
    
    print(f"\nОптимальный порог: {optimal_threshold:.4f}")
    print(f"\nМетрики при оптимальном пороге:")
    print(f"  - Recall:    {optimal_metrics['recall_optimal']:.4f}")
    print(f"  - Precision: {optimal_metrics['precision_optimal']:.4f}")
    print(f"  - F1-Score:  {optimal_metrics['f1_optimal']:.4f}")

Базовый scale_pos_weight: 3.91
Параметры для поиска:
  - iterations: [300, 500, 700]
  - depth: [4, 6, 8]
  - learning_rate: [0.01, 0.05, 0.1]
  - l2_leaf_reg: [1, 3, 5]
  - border_count: [64, 128, 255]
  - class_weights: [[1, 1], [1, 2], [1, 4], [1, 6], [1, 3]]

Запуск Grid Search для CatBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
  - learning_rate: 0.01
  - l2_leaf_reg: 1
  - iterations: 700
  - depth: 4
  - class_weights: [1, 3]
  - border_count: 64

Лучший CV ROC-AUC: 0.8510

Метрики на тестовой выборке (порог 0.5):
  - Accuracy:  0.8280
  - Precision: 0.5631
  - Recall:    0.6904
  - F1-Score:  0.6203
  - ROC-AUC:   0.8614
  - Итераций:  0

Оптимальный порог: 0.5300

Метрики при оптимальном пороге:
  - Recall:    0.6732
  - Precision: 0.5944
  - F1-Score:  0.6313
🏃 View run catboost_gridsearch_balanced at: http://localhost:5000/#/experiments/1/runs/0f7f3861528e4ed2bc206b1a94469c54
🧪 View experiment at: http://localhost:5000/#/experiments/1
